## Robinhood question: 

### <b>Referral count </b>

Robinhood is famous for its referral program. It’s exciting to see our users spreading the word across their friends and family. One thing that is interesting about the program is the network effect it creates. We would like to build a dashboard to track the status of the program. Specifically, we would like to learn about how people refer others through the chain of referral.

For the purpose of this question, we consider that a person refers all other people down the referral chain. For example, A refers B, C, and D in a referral chain of A -> B -> C -> D. Please build a leaderboard for the top 3 users who have the most referred users along with the referral count.

Referral rules:

A user can only be referred once.
Once the user is on the RH platform, he/she cannot be referred by other users. For example: if A refers B, no other user can refer A or B since both of them are on the RH platform.
Referrals in the input will appear in the order they were made.
Leaderboard rules:

The user must have at least 1 referral count to be on the leaderboard.
The leaderboard contains at most 3 users.
The list should be sorted by the referral count in descending order.
If there are users with the same referral count, break the ties by the alphabetical order of the user name.

input: [["A", "B"], ["A", "C"], ["C", "D"], ["E", "F"],["E", "G"]]  
middle: {"A": 3, "B": 0, "C" 1, "D": 0, "E": 2, "F": 0, "G": 0}  
output: ["A", "E", "C"]

time complexity:
- constructing graph O(E), length of input referrals is actually the edge number 
- dfs with a dictionary only traverse all vertice once so O(V)
- when doing the sort at the end, for totall V nodes, the sort cost O(VlogV)
- so total is O(E)+O(V)+O(VlogV)
- since the referral can happen only once, so E, V all related to the length of referral N. and dominant by the sorting step, so totally O(NlogN)

In [69]:
from collections import defaultdict
from typing import List
def getNumberOfBacklogOrders(referrals:List[List[str]]):
    # construct a graph, like {"A": ["B", "C"], "B": ["D"]}
    graph = defaultdict(list)
    for refer, friend in referrals:
        graph[refer].append(friend)

    # store the refer count map like {"A": 3, "B": 1}
    count_map = defaultdict(int)
    def dfs_helper(node):
        # return the count of each node.
        # parent count is its children count + 1
        if len(graph[node]) == 0:
            count_map[node] == 0
            return 0
        if node in count_map:
            return count_map[node]
        count = 0
        for sub_node in graph[node]:
            count += dfs_helper(sub_node) + 1
        count_map[node] = count
        return count
    for item in list(graph.keys()):
        dfs_helper(item) # populate the count_map
    print(count_map)
    return sorted(count_map, key=lambda x: (-count_map[x], x))[:3]



In [2]:
getNumberOfBacklogOrders( [["A", "B"], ["A", "C"], ["C", "D"], ["E", "F"],["E", "G"]])

defaultdict(<class 'int'>, {'B': 0, 'D': 0, 'C': 1, 'A': 3, 'F': 0, 'G': 0, 'E': 2})


['A', 'E', 'C']

In [3]:
a = {"D":0, "E" :2, "A":3, "B": 2}
sorted(a, key=lambda x: (-a[x], x))

['A', 'B', 'E', 'D']

In [4]:
getNumberOfBacklogOrders([["A", "B"], ["A", "C"], ["C", "D"]])

defaultdict(<class 'int'>, {'B': 0, 'D': 0, 'C': 1, 'A': 3})


['A', 'C', 'B']

### <b> house trades and street trades </b>

A trade is defined as a fixed-width string containing the 4 properties given below separated by commas:  
```
Symbol (4 alphabetical characters, left-padded by spaces)
Type (either "B" or "S" for buy or sell)
Quantity (4 digits, left-padded by zeros)
ID (6 alphanumeric characters)
```
e.g.
"AAPL,B,0100,ABC123"

which represents a trade of a buy of 100 shares of AAPL with ID "ABC123"

Given two lists of trades - called "house" and "street" trades, write code to filter out groups of matches between trades and return a list of unmatched house and street trades sorted alphabetically. There are many ways to match trades, the first and most important way is an exact match (Tests 1-5):

An exact match is a house_trade+street_trade pair with identical symbol, type, quantity, and ID


Note: Trades are distinct but not unique

For example, given the following input:

// house_trades:  
[  
"AAPL,B,0100,ABC123",  
"AAPL,B,0100,ABC123",  
"GOOG,S,0050,CDC333"  
]

// street_trades:  
[  
" FB,B,0100,GBGGGG",  
"AAPL,B,0100,ABC123"  
]

We would expect the following output:  

[  
" FB,B,0100,GBGGGG",  
"AAPL,B,0100,ABC123",  
"GOOG,S,0050,CDC333"  
]

Because the first (or second) house trade and second street trade form an exact match, leaving behind three unmatched trades.


should get the overlapped_count, cannot assume the common count is 1.  
time complexity:
- counter action for two list takes O(N1+N2) given N1, N2 are the length of the two lists
- & for set operation takes O(N3) N3 is the min of (N1, N2)
- loop over the common stack, inside of the loop, all actions takes O(1)
- merge two dictionary takes O(N)
- sorting the dictioary takes O(KlogK) suppose k <= N1+N2
- so total time complexity is O(N1)+O(N2)+O(KlogK), in worse casae K is close to N1+N2, so can be simplied as O(NlogN)  N is porpotional to the length of the 2 list.

In [107]:
from typing import List
from collections import Counter
from functools import reduce
def house_and_street_trades(house_trades: List[str], street_trades: List[str]):
    house_trade_count_map = Counter(house_trades)
    street_trade_count_map = Counter(street_trades)
    house_trade_set = set(house_trades)
    street_trade_set = set(street_trades)
    common = house_trade_set & street_trade_set
    for stock in common:
        common_count = min(house_trade_count_map[stock], street_trade_count_map[stock])
        house_trade_count_map[stock] -= common_count # -> cannot assume only 1 is the count in common
        if house_trade_count_map[stock] == 0:
            del house_trade_count_map[stock]
        street_trade_count_map[stock] -= common_count
        if street_trade_count_map[stock] == 0:
            del street_trade_count_map[stock]
    # merge two map into one
    house_trade_count_map.update(street_trade_count_map)
    return reduce(lambda x, y: x+y, [ [k] * house_trade_count_map[k]] for k in sorted(house_trade_count_map)])
#     res = reduce(lambda x, y: x+y, [[i]*v for i, v in  house_trade_count_map.items()]) + reduce(lambda x, y: x+y, [[i]*v for i, v in  street_trade_count_map.items()])
#     return sorted(res)

In [108]:
house_trades=["AAPL,B,0100,ABC123", 
 "GOOG,S,0050,CDC333"]
street_trades=["FB,B,0100,GBGGGG", 
 "AAPL,B,0100,ABC123"]
house_and_street_trades(house_trades, street_trades)

['FB,B,0100,GBGGGG', 'GOOG,S,0050,CDC333']

In [109]:
house_trades= ["AAPL,B,0010,ABC123", 
 "AAPL,S,0015,ZYX444", 
 "AAPL,S,0015,ZYX444", 
 "GOOG,S,0050,GHG545"]
street_trades =["GOOG,S,0050,GHG545", 
 "AAPL,S,0015,ZYX444", 
 "AAPL,B,0500,TTT222"]
house_and_street_trades(house_trades, street_trades)

['AAPL,B,0010,ABC123', 'AAPL,B,0500,TTT222', 'AAPL,S,0015,ZYX444']

In [101]:
house_trades= ["AAPL,B,0010,ABC123", 
 "AAPL,S,0015,ZYX444", 
 "AAPL,S,0015,ZYX444", 
 "GOOG,S,0050,GHG545"]
street_trades =["GOOG,S,0050,GHG545", 
 "AAPL,S,0015,ZYX444", 
 "AAPL,S,0015,ZYX444", 
 "AAPL,B,0500,TTT222"]
house_and_street_trades(house_trades, street_trades)

['AAPL,B,0010,ABC123', 'AAPL,B,0500,TTT222']

In [9]:
from functools import reduce
a = {"hey": 2, "name": 1}
reduce(lambda x,y: x+y, [[i]*v for i, v in a.items()])

['hey', 'hey', 'name']

#### Follow-up 1 (Test 6,7,8,9): 

A "fuzzy" match is a house_trade+street_trade pair with identical symbol, type, and quantity ignoring ID. Prioritize exact matches over fuzzy matches. Prioritize matching the earliest alphabetical house trade with the earliest alphabetical street trade in case of ties.

In [120]:
from typing import List
from collections import Counter

def house_and_street_trades(house_trades: List[str], street_trades: List[str]):
    filtered_house_trades, filtered_street_trades = exact_match(house_trades, street_trades)
    filtered_house_trades, filtered_street_trades = fuzzy_match(filtered_house_trades, filtered_street_trades)
    return sorted(filtered_house_trades + filtered_street_trades)

def exact_match(house_trades: List[str], street_trades: List[str]):
    house_trade_count_map = Counter(house_trades)
    street_trade_count_map = Counter(street_trades)
    house_trade_set = set(house_trades)
    street_trade_set = set(street_trades)
    common = house_trade_set & street_trade_set
    for stock in common:
        common_count = min(house_trade_count_map[stock], street_trade_count_map[stock])
        house_trade_count_map[stock] -= common_count # -> cannot assume only 1 is the count in common
        if house_trade_count_map[stock] == 0:
            del house_trade_count_map[stock]
        street_trade_count_map[stock] -= common_count
        if street_trade_count_map[stock] == 0:
            del street_trade_count_map[stock]
    filtered_house_trades = reduce(lambda x,y: x+y, [[t]*house_trade_count_map[t] for t in sorted(house_trade_count_map)])
    filtered_street_trades = reduce(lambda x,y: x+y, [[t]*street_trade_count_map[t] for t in sorted(street_trade_count_map)])
    return filtered_house_trades, filtered_street_trades

def is_fuzzy_match(s1, s2):
    ind = s1.rfind(",")
    return s2.startswith(s1[:ind])

def fuzzy_match(house_trades: List[str], street_trades: List[str]):
    # still O(N) since the two list only visited once
    
    # can be omitted since exact match already sort the result
    house_trades = sorted(house_trades)
    street_trades = sorted(street_trades)
    
    filtered_house_trades = []
    filtered_street_trades = []
    
    i, j = 0, 0
    while i < len(house_trades) and j < len(street_trades):
        house_t = house_trades[i]
        street_t = street_trades[j]
        if is_fuzzy_match(house_t, street_t):
            i+=1
            j+=1
#             continue
        elif house_t<street_t:
            i += 1
            filtered_house_trades.append(house_t)
        elif street_t < house_t:
            j += 1
            filtered_street_trades.append(street_t)
    if i < len(house_trades):
        filtered_house_trades += house_trades[i:]
    if j < len(street_trades):
        filtered_street_trades += street_trades[j:]
    return filtered_house_trades, filtered_street_trades
    

In [114]:
house_trades = ["A,3", "A,3", "A,4", "A,4", "A,4", "B,1"]
street_trades = ["A,3", "A,5", "C,1"]

In [115]:
exact_match(house_trades, street_trades)

(['A,3', 'A,4', 'A,4', 'A,4', 'B,1'], ['A,5', 'C,1'])

In [121]:
fuzzy_match(['A,3', 'A,4', 'A,4', 'A,4', 'B,1'], ['A,5', 'C,1'])

(['A,4', 'A,4', 'A,4', 'B,1'], ['C,1'])

In [122]:
house_and_street_trades(house_trades, street_trades)

['A,4', 'A,4', 'A,4', 'B,1', 'C,1']

In [24]:
house_trades=["AAPL,B,0100,ABC123", 
 "GOOG,S,0050,CDC333"]
street_trades=["FB,B,0100,GBGGGG", 
 "AAPL,B,0100,ABC123"]
exact_match_filter(house_trades, street_trades)

(['GOOG,S,0050,CDC333'], ['FB,B,0100,GBGGGG'])

In [27]:
fuzzy_match_filter(house_trades, street_trades)

['FB,B,0100,GBGGGG', 'GOOG,S,0050,CDC333']

### <b>  margin call, trading system </b>

Input will be a list of trades, each of which is itself a list of strings in the form [timestamp, symbol, B/S (for buy/sell), quantity, price], e.g.

[["1", "AAPL", "B", "10", "10"], ["3", "GOOG", "B", "20", "5"], ["10", "AAPL", "S", "5", "15"]]

**Input assumptions:**

    - The input is sorted by timestamp
    - All numerical values are nonnegative integers
    - Trades will always be valid (i.e. a customer will never sell more of a stock than they own).
#### step 1: building cusotmer portfolio

 From the provided list of trades, our goal is to maintain the customer's resulting portfolio (meaning everything they own), assuming they begin with $1000. 
 
For instance, in the above example, the customer would end up with $875, 5 shares of AAPL, and 20 shares of GOOG. You should return a list representing this portfolio, formatting each individual position as a list of strings in the form [symbol, quantity], using 'CASH' as the symbol for cash and sorting the remaining stocks alphabetically based on symbol. For instance, the above portfolio would be represented as  

    [["CASH", "875"], ["AAPL", "5"], ["GOOG", "20"]]  

input: 
    [["1", "AAPL", "B", "10", "10"], ["3", "GOOG", "B", "20", "5"], ["10", "AAPL", "S", "5", "15"]]

output:
    [["CASH", "875"], ["AAPL", "5"], ["GOOG", "20"]]  

In [124]:
def setup_portfolio(trades: List[List[str]]):
    cash = 1000
    stock_pool = defaultdict(int) # {symbol: num}
    for timestamp, symbol, trade_type, num, price in trades:
        if trade_type == "B":
            # buy stock
            cash -= int(num)*int(price)
            stock_pool[symbol] += int(num)
        if trade_type == "S":
            # sell stock, assume trade is always valid
            cash += int(num)*int(price)
            stock_pool[symbol] -= int(num)
            
    portfolio = [["CASH", str(cash)]]
    for stock in sorted(stock_pool.keys()):
        if stock_pool[stock]!= 0:
            portfolio.append([stock, str(stock_pool[stock])])
    return portfolio
        

In [125]:
trades = [["1", "AAPL", "B", "10", "10"], ["3", "GOOG", "B", "20", "5"], ["10", "AAPL", "S", "5", "15"]]
setup_portfolio(trades)

[['CASH', '875'], ['AAPL', '5'], ['GOOG', '20']]

#### step 2:  margin call

If the customer ever ends up with a negative amount of cash **after a buy**, they then enter a process known as a **margin call** to correct the situation. In this process, we forcefully sell stocks in the customer's portfolio (sometimes including the shares we just bought) until their cash becomes non-negative again.

We sell shares from the most expensive to least expensive shares (based on each symbol's most-recently-traded price) with ties broken by preferring the alphabetically earliest symbol. Assume we're able to sell any number of shares in a symbol at that symbol's most-recently-traded price.

For example, for this input:

    ```
    [["1", "AAPL", "B", "10", "100"],
    ["2", "AAPL", "S", "2", "80"],
    ["3", "GOOG", "B", "15", "20"]]

    ```


The customer would be left with 8 AAPL shares, 15 GOOG shares, and 80 a share) to cover the deficit. Afterwards, they would have 6 shares of AAPL, 15 shares of GOOG, and a cash balance of $20.

process:
- from the above input, the customer porfolio will be 
 - [["CASH", "-140"], ["AAPL", "8"], ["GOOG", "15]]
 - and AAPL recent price is 80, AAPL has higer price than GOOG
 - so for margin call, sell 2 AAPL at $80 to cover the negative CACH flow
 - finall get [["CASH", "20"], ["AAPL", "6"], ["GOOG", "15]]
 

In [37]:
from collections import deque
def get_recent_price(trades: List[List[str]]):
    stock_price_map = defaultdict(int)
    for timestamp, symbol, trade_type, num, price in trades:
        stock_price_map[symbol] = int(price) # get the most recent price, given that trades is sorted by time
    return stock_price_map  
        
def margin_call(stock_prices: dict[str, int], portfolio: List[List[str]]):
    cash = int(portfolio[0][1])
    
    if cash < 0:
        portfolio_map = {stock: int(num) for stock, num in portfolio[1:]}
        # sort portfolio by price
        stock_sorted_by_price = deque(sorted(stock_prices, key=lambda x: -stock_prices[x]))
        while cash < 0 and stock_sorted_by_price:
            stock = stock_sorted_by_price[0]
            cash += stock_prices[stock]
            portfolio_map[stock] -= 1
            if portfolio_map[stock]  == 0:
                del portfolio_map[stock]
                stock_sorted_by_price.popleft()
        new_portfolio = [["CASH", str(cash)]]
        for stock in sorted(portfolio_map.keys()):
            new_portfolio.append([stock, str(portfolio_map[stock])])
        return new_portfolio
    return portfolio
    
    

In [40]:
trades = [["1", "AAPL", "B", "10", "100"],
["2", "AAPL", "S", "2", "80"],
["3", "GOOG", "B", "15", "20"]]
porfolio = setup_portfolio(trades)
print(porfolio)
recent_price_map = get_recent_price(trades)
print(recent_price_map)
new_portfolio = margin_call(recent_price_map, porfolio)
print("new_portfolio ", new_portfolio)

[['CASH', '-140'], ['AAPL', '8'], ['GOOG', '15']]
defaultdict(<class 'int'>, {'AAPL': 80, 'GOOG': 20})
new_portfolio  [['CASH', '20'], ['AAPL', '6'], ['GOOG', '15']]


In [130]:
# if the step 1 returns {stock: count} map and cash value

# def setup_portfolio(trades: List[List[str]]):
    # call a helper function that returns {stock: count} map and cash value
    # return a list of [str]

# def get_recent_price(trades: List[List[str]]):
#     stock_price_map = defaultdict(int)
#     for timestamp, symbol, trade_type, num, price in trades:
#         stock_price_map[symbol] = int(price) # get the most recent price, given that trades is sorted by time
#     return stock_price_map  

def margin_call(portfolio_map: dict[str, int], stock_recent_price_map:dict[str,int], cash:int):
    stock_by_price = sorted(stock_recent_price_map, key=lambda x: -stock_recent_price_map[x])
    print("stock_by_price", stock_by_price)
    for stock in stock_by_price:
        if stock in portfolio_map:
            while cash < 0 and portfolio_map[stock] > 0:
                cash += stock_recent_price_map[stock]
                portfolio_map[stock] -= 1
                if portfolio_map[stock] == 0:
                    del portfolio_map[stock]
            if cash > 0:
                break
    return portfolio_map, cash
    

In [131]:
margin_call({"a": 4, "b":5, "c":2}, {"a":2, "b":5, "c":9, "d": 10}, -3)

stock_by_price ['d', 'c', 'b', 'a']


({'a': 4, 'b': 5, 'c': 1}, 6)

#### step 3: 

 Certain stocks have special classifications, and require the customer to also own another "collateral" stock, meaning it cannot be sold during the margin call process. Our goal is to handle a simplified version of this phenomenon.
 
Formally, we'll consider stocks with symbols ending in "O" to be special, with the remainder of the symbol identifying its collateral stock. For example, AAPLO is special, and its collateral stock is AAPL. **At all times**, the customer must hold at least as many shares of the collateral stock as they do the special stock; e.g. they must own at least as many shares of AAPL as they do of AAPLO. 
 - some special stocks cannot be sold in a normal way

As a result, the margin call process will now sell the most valuable **non-collateral** share until the balance is positive again. Note that if this sells a special stock, some of the collateral stock may be freed up to be sold.
 - if sell x AAPLO, should sell X AAPL too
 
 for example  
 input: [["1", "AAPL", "B", "5", "100"], ["2", "GOOG", "B", "5", "75"], ["3", "AAPLO", "B", "5", "50"]]  
 portfolio will be: [["CASH", -125], ["AAPL", 5], ["GOOG", 5], ["AAPLO", 5]]
 
 after the special rule, although AAPL has higher price, GOOG should be sold first at price $75. so end up we have  
 portfolio: [["CASH", "25"], ["AAPL", "5"], ["AAPLO", "5"], ["GOOG", "3"]

In [158]:
# call step_1 function to get stock_count_dict, cash
def step_3(stock_count_dict, stock_recent_price_map:dict[str,int], cash, trades):
    special_to_collateral_map = {} # stock ending with 'O' is considered special
    collateral_to_special_map = {} # non sellable
    
    for timestamp, symbol, trade_type, num, price in trades:
        if symbol.endswith("O") and symbol in stock_count_dict:
            # collateral stock should have same count as special_stock
            collateral_stock = symbol[:-1]
            special_to_collateral_map[symbol] = collateral_stock
            collateral_to_special_map[collateral_stock] = symbol
            if stock_count_dict[symbol] > stock_count_dict[collateral_stock]:
                # buy additional collateral_stock:
                amount = stock_count_dict[symbol]  - stock_count_dict[collateral_stock]
                cash -= stock_recent_price_map[collateral_stock] * amount
                stock_count_dict[collateral_stock] += amount
                print("here", stock_count_dict, cash)
            elif stock_count_dict[symbol]  < stock_count_dict[collateral_stock]:
                # sell additional collateral_stock:
                amount = stock_count_dict[symbol] - stock_count_dict[collateral_stock]
                cash += stock_recent_price_map[collateral_stock]*amount
                stock_count_dict[collateral_stock] -= amount
                
    # now with updated cash and stock_count_dict 
    print("updated stock", stock_count_dict, cash)
    print("special_to_collateral_map", special_to_collateral_map)
    print("collateral_to_special_map", collateral_to_special_map)
    # margin call
    if cash < 0:
        stock_by_price = sorted(stock_recent_price_map, key=lambda x: -stock_recent_price_map[x])

        for stock in stock_by_price:
            if stock in stock_count_dict and (stock not in collateral_to_special_map):
                while cash < 0 and stock_count_dict[stock] > 0:
                    cash += stock_recent_price_map[stock]
                    stock_count_dict[stock] -= 1
                    if stock_count_dict[stock] == 0:
                        del stock_count_dict[stock]
                    if stock in special_to_collateral_map:
                        # free up non sellable stock
                        collateral_stock = special_to_collateral_map[stock]
                        cash += stock_recent_price_map[collateral_stock]
                        stock_count_dict[collateral_stock] -=1
                        if stock_count_dict[collateral_stock] == 0:
                            del stock_count_dict[collateral_stock]
                if cash > 0:
                    break
    return stock_count_dict, cash
                

In [159]:
stock_count_dict = {"a": 3, "b": 5, "aO": 5}
stock_recent_price_map = {"a": 30, "b":25, "aO": 20, "d": 50}
cash = -20
trades_list = [["1", "a", "B", "3", "30"], ["2", "b", "B", "5", "25"], ["3", "aO", "B", "5", "20"]]
step_3(stock_count_dict, stock_recent_price_map, cash, trades_list)

here {'a': 5, 'b': 5, 'aO': 5} -80
updated stock {'a': 5, 'b': 5, 'aO': 5} -80
special_to_collateral_map {'aO': 'a'}
collateral_to_special_map {'a': 'aO'}


({'a': 5, 'b': 1, 'aO': 5}, 20)

### <b> service dependency dag </b>

Overview  
You are building an application that consists of many different services that can depend on each other. One of these services is the entrypoint which receives user requests and then makes requests to each of its dependencies, which will in turn call each of their dependencies and so on before returning.

Given a directed acyclic graph that contains these dependencies, you are tasked with determining the "load factor" for each of these services to handle this load. The load factor of a service is defined as the number of units of load it will receive if the entrypoint receives a 1 unit of load. Note that we are interested in the worst case capacity. For a given downstream service, its load factor is the number of units of load it is required to handle if all upstream services made simultaneous requests. For example, in the following dependency graph where A is the entrypoint:

- Input:
service_list = ["logging=",   
 "user=logging",   
 "orders=user,foobar",   
 "recommendations=user,orders",   
 "dashboard=user,orders,recommendations"]  
entrypoint = "dashboard"  

- Output (note sorted by service name)
["dashboard*1",   
 "logging*4",   
 "orders*2",   
 "recommendations*1",   
 "user*4"]  
 

In [ ]:
from collections import deque, defaultdict
def solution(service_list, entrypoint):
    service_graph = defaultdict(list) # key: starting service, val: [dependency services]
    for service in service_list:
        # parse the service_list to populate the graph
        starting_service, dependent_services = service.split("=")
        service_graph[starting_service] += dependent_services.split(",")
    
    queue_service = deque(service_graph[entrypoint])
    result_map = defaultdict(int) # key: service, val: count
    result_map[entrypoint] = 1
    while queue_service:
        service = queue_service.popleft()
        result_map[service] += 1
        depedent_services = service_graph[service]
        for s in depedent_services:
            if s in service_graph:
                queue_service.append(s)
    
    sorted_result = sorted(result_map)
    return [f"{service}*{result_map[service]}" for service in sorted_result]

simplified version:  
- input: [A: B, B: C, C:D, A:C], entry point A.   
- output: [A: 1, B:2, C:2, D:2]
- time complexity:
 - loop over the service list, each of which is an edge. so O(E)
 - when loop over Q, it is actually loop over each edge, so O(E)
 - finally need to sort the result dictionaly, which is O(VLogV) V is the number of vertex.
 so total time complexity is O(E) + O(VlogV)

In [51]:
from collections import deque, defaultdict
from typing import List
def find_service_count(service_list: List[str], entry_point:str):
    # construct a service dependency graph: {service: [dependent_s1, dependent_s2]}
    service_graph = defaultdict(list)
    for services in service_list:
        service, dependent_service = services.split(",")
        service_graph[service].append(dependent_service)
        
    # result dict to save the {service: count}
    res_dict = defaultdict(int)
    res_dict[entry_point] += 1
    # a deque to keep track of the service visited
    q = deque(service_graph[entry_point])
    while q:
        service = q.pop()
        res_dict[service] +=1 
        for dependent_service in service_graph[service]:
            q.append(dependent_service)
    res_dict = dict(sorted(res_dict.items()))
    return res_dict

In [52]:
find_service_count(["A,B", "B,C", "C,D", "A,C"], "A")

service_graph defaultdict(<class 'list'>, {'A': ['B', 'C'], 'B': ['C'], 'C': ['D']})


{'A': 1, 'B': 1, 'C': 2, 'D': 2}

### <b> 1801 number of orders in backlog </b>

orders[i] = [pricei, amounti, orderTypei]  where orderType = 0 means buy, 1 means sell.  Return the total amount of orders in the backlog after placing all the orders from the input.  
- If the order is a buy order, you look at the sell order with the smallest price in the backlog. If that sell order's price is smaller than or equal to the current buy order's price, they will match and be executed, and that sell order will be removed from the backlog. Else, the buy order is added to the backlog.
- Vice versa, if the order is a sell order, you look at the buy order with the largest price in the backlog. If that buy order's price is larger than or equal to the current sell order's price, they will match and be executed, and that buy order will be removed from the backlog. Else, the sell order is added to the backlog.

input: orders = [[10,5,0],[15,2,1],[25,1,1],[30,4,0]]  
output: 6

time complexity:
- every heappush operation is O(LogK) K is the length of the queue size, for all items, the heappush cost O(NLogN) N is the total item number of the orders list
- in the while loop, the heappop operation may occur one or twice for each iteration, in worse case all items are popped, each heappop cost O(LogK), and will happen k times, k is the length of the queue.
- so totally it is O(NlogN) N is the totally number of the order list

In [67]:
from typing import List
from heapq import heappop, heappush
def getNumberOfBacklogOrders(orders: List[List[int]]):
    # 2 heap queue
    buy_orders = [] # (-p, amount)
    sell_orders =[] # (p, amount)
    for p, amount, t in orders:
        if t == 0:
            # buy order
            heappush(buy_orders, [-p, amount])
        if t == 1:
            # sell order
            heappush(sell_orders, [p, amount])
        while buy_orders and sell_orders and -1*buy_orders[0][0] >= sell_orders[0][0]:
            # if current smallest sell offer price <= largest buy offer price
            transaction_amount = min(buy_orders[0][1], sell_orders[0][1])
            buy_orders[0][1] -= transaction_amount
            sell_orders[0][1] -= transaction_amount
            if buy_orders[0][1] == 0: 
                heappop(buy_orders)
            if sell_orders[0][1] == 0:
                heappop(sell_orders)
    return sum(amount for p, amount in buy_orders) + sum(amount for p, amount in sell_orders) % (10**9 + 7)
    

In [68]:
getNumberOfBacklogOrders([[10,5,0],[15,2,1],[25,1,1],[30,4,0]])

6

#### <b> 1656 design an ordered stream </b>

the problem is not well designed

Input:  
["OrderedStream", "insert", "insert", "insert", "insert", "insert"]
[[5], [3, "ccccc"], [1, "aaaaa"], [2, "bbbbb"], [5, "eeeee"], [4, "ddddd"]]  
Output  
[null, [], ["aaaaa"], ["bbbbb", "ccccc"], [], ["ddddd", "eeeee"]]  


In [71]:
class OrderedStream:

    def __init__(self, n: int):
        self.data = [None] * n
        self.pointer = 0

    def insert(self, idKey: int, value: str) -> List[str]:
        idKey-= 1
        self.data[idKey] = value
        if idKey > self.pointer: return []
        while self.pointer < len(self.data) and self.data[self.pointer]:
            self.pointer += 1
        return self.data[idKey: self.pointer]

### course schedule series

#### 207 source schedule 

construct a graph to store the {pre: [next_course]}, and a degree list to keep track of how mnay prerequisites each course has. Since the course is 0 to n-1, the degree var can be simplied as a list.

prerequisites[i] = [ai, bi] indicates that you must take course bi first if you want to take course ai.
so bi is ai's prerequisite.  
time complexity:
- construct the graph need to loop over the prerequisites list, with length M, so O(M)
- identify the starting course takes O(N)
- the queue will loop over the connection (edge) between pre - next course only once, which is O(M)
- so total time complexity is O(M)+O(N) M is the length of prerequisites, N is the course numbers

In [77]:
from collections import defaultdict, deque
def canFinish(numCourses: int, prerequisites: List[List[int]]) -> bool:
    # construct a graph:{prerequisite: [dependings]}
    graph = defaultdict(list)
    # construct a degree dictionary: {course: degree integer}
    # n degree means this course has n prerequisits need to take
    degree = [0]*numCourses
    for d, pre in prerequisites:
        graph[pre].append(d)
        degree[d]+=1 # course number happens to be used as index

    # the course with degree 0 is the able to take first. if no course has 0 degree, the courses are in a loop, cant be finished
    start_courses = deque([i for i in range(numCourses) if degree[i]==0])
    while start_courses:
        pre = start_courses.popleft()
        for j in graph[pre]:
            # j is the depending courses of i
            degree[j]-=1  # number of requisites needs to be finished before able to take the course j is reduced by 1
            if degree[j]==0:
                # able to take it as next course
                start_courses.append(j)
    # if all courses are taken, they will all be 0
    return not sum(degree)

In [78]:
canFinish(2, [[1,0]])

True

#### 210 course schedule II 

Return the ordering of courses you should take to finish all courses. If there are many valid answers, return any of them. If it is impossible to finish all courses, return an empty array.

same as 207 but adding a res list to keep track of the courses taken

time complexity, same as 207
- construct the graph: O(M)
- find starting course: O(M)
- loop over each edge: O(M)
- total O(M)+O(N)

In [86]:
from collections import defaultdict, deque

def findOrder(numCourses: int, prerequisites: List[List[int]]) -> List[int]: 
    graph = defaultdict(list)  # {prerequisite: [depending courses]}
    degree=[0]*numCourses  # [integer] how many prerequisites needs to take

    for course, pre in prerequisites:
        graph[pre].append(course)
        degree[course]+=1 

    result=[]
    # start with course with no prerequisite
    starting_course = deque([i for i in range(numCourses) if degree[i]==0])
    while starting_course:
        course = starting_course.popleft()
        result.append(course)
        # its dependencies' degree - 1
        for d in graph[course]:
            degree[d]-=1
            if degree[d]==0:
                # able to be taken next
                starting_course.append(d)

    if sum(degree)==0:
        # all courses are taken
        return result
    else:
        return []

In [87]:
findOrder(2, [[1,0]])

[0, 1]

In [88]:
findOrder(4, [[1,0],[2,0],[3,1],[3,2]])

[0, 1, 2, 3]

#### 630 course schedule III 

There are n different online courses numbered from 1 to n. You are given an array courses where courses[i] = [durationi, lastDayi] indicate that the ith course should be taken continuously for durationi days and must be finished before or on lastDayi.

You will start on the 1st day and you cannot take two or more courses simultaneously.

Return the maximum number of courses that you can take.

time complexity:
- sort the courses list takes O(NlogN) N is the length of courses
- for loop of course list takes O(N)
- heappush takes O(LogN) per per time at worse case, for N iteration, it is O(NlogN)
- heappop takes O(1)
- so total is O(NLogN)

In [90]:
import heapq
def scheduleCourse( courses: List[List[int]]) -> int:
    # sort the courses list by lastDay
    courses = sorted(courses, key =lambda x: x[1])
    pq = [] # keep track of the duration of courses taken
    start = 0
    for duration, end in courses:
        start += duration
        # course with longer duration will be on top of queue
        heapq.heappush(pq, -duration)
        # when the course added not fit the criteria
        # greedily remove the largest-length cours, thanks to the heapq to keep check what the longest course is
        # by removing the course taking the longest time, can make sure the length of courses taken is the maximal
        while start > end:
            # because the time now because a negative value
            # poping out meanings not taking that course
            start += heapq.heappop(pq)
    return len(pq)

In [91]:
scheduleCourse([[100,200],[200,1300],[1000,1250],[2000,3200]])

3

In [92]:
scheduleCourse([[3,2],[4,3]])

0

giving a stream of inputs return the maximum committeable values

input : [ 2, 0, 1]   
output: [-1, -0, 2]

In [166]:
from heapq import heapify, heappush, heappop
def commit_stream(offsets: List[int]):
    uncommitted = [] # pqueue ascending order
    committed = -1 # keep track of largest committed value for far
    result = []
    for i in offsets:
        if i == committed + 1:
            # i is considerable
            commitable = i
            while uncommitted:
                # loop over the uncommited queue to find the largest commitable value
                smallest_uncommitted_value = uncommitted[0]
                if smallest_uncommitted_value == commitable + 1:
                    commitable = smallest_uncommitted_value
                    heappop(uncommitted)
                else:
                    break
            committed = commitable
            result.append(commitable)
        else:
            result.append(-1)
            heappush(uncommitted, i)
    return result
                
            
        
        

In [167]:
commit_stream([2,1,0,5,4])

[-1, -1, 2, -1, -1]

In [164]:
commit_stream([0, 1, 2])

[0, 1, 2]

In [165]:
commit_stream([3,1,2,0])

[-1, -1, -1, 3]